In [ ]:
!pip install autotrain-advanced
!pip install huggingface_hub

In [ ]:
!autotrain setup --update-torch

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_qa_bot = AutoModelForCausalLM.from_pretrained("ApurvaKolhe/newsQA")
model_qa_bot.eval()

In [ ]:
tokenizer_qa_bot = AutoTokenizer.from_pretrained("ApurvaKolhe/newsQA")

In [ ]:
from transformers import T5Tokenizer, pipeline, TFAutoModelForSeq2SeqLM
model_summarizer = TFAutoModelForSeq2SeqLM.from_pretrained("ApurvaKolhe/text_summarization_t5")
tokenizer_summarizer = T5Tokenizer.from_pretrained('t5-small')

In [ ]:
generation_config = model_qa_bot.generation_config
generation_config.max_new_tokens = 50
generation_config.temperature = 0.5
generation_config.top_p = 0.5
generation_config.num_return_sequences = 1
generation_config.do_sample = True

In [ ]:
# QA model smoke test
test_prompt = f"""[INST] Answer the given question based on the news article below : Article: After years of speculation and rare case reports , a study suggests that stimulant medication -- mostly used to treat attention deficit hyperactivity disorder -- may have played a role in a handful of cases of sudden , unexplained death in children and adolescents . Untreated ADHD can lead to poor school performance and increase adolescents ' risk for harmful behavior . The study authors stress , however , that parents and doctors should not refrain from treating children with ADHD just because of these results . `` The association is significant in that it 's real , but that does n't mean it 's not a very low risk , '' says lead author Madelyn S. Gould , Ph.D. , a professor of psychiatry and public health at Columbia University , in New York . [/INST]
Sure! What is your question?
[INST] QUESTION : Who is the lead author of the study ? [/INST]
"""
input_ids = tokenizer_qa_bot.encode(test_prompt, return_tensors='pt')
output = model_qa_bot.generate(input_ids, temperature=0.5)
print(tokenizer_qa_bot.decode(output[0]))

In [ ]:
import nltk
nltk.download('punkt')

def chunk_text_by_sentences(text, max_token_length=512):
    sentences = nltk.tokenize.sent_tokenize(text)
    current_chunk = ""
    chunks = []

    for sentence in sentences:
        token_length = len(tokenizer_summarizer.encode(current_chunk + sentence))
        if token_length < max_token_length:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk)
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [ ]:
# T5 summarizer smoke test
pipe = pipeline("summarization", model=model_summarizer, tokenizer=tokenizer_summarizer, framework="tf")
test_chunks = chunk_text_by_sentences("After years of speculation and rare case reports , a study suggests that stimulant medication -- mostly used to treat attention deficit hyperactivity disorder -- may have played a role in a handful of cases of sudden , unexplained death in children and adolescents . Untreated ADHD can lead to poor school performance and increase adolescents ' risk for harmful behavior . The study authors stress , however , that parents and doctors should not refrain from treating children with ADHD just because of these results . `` The association is significant in that it 's real , but that does n't mean it 's not a very low risk , '' says lead author Madelyn S. Gould , Ph.D. , a professor of psychiatry and public health at Columbia University , in New York .")
test_summaries = [pipe(chunk)[0]['summary_text'] for chunk in test_chunks]
print(" ".join(test_summaries))

In [ ]:
!pip install gradio

In [ ]:
def generate_prompt_for_qa(question, article_text):
    prompt = f"""[INST] Answer the given question based on the news article below : Article: {article_text} [/INST]
    Sure! What is your question? [INST] QUESTION : {question} ? [/INST]"""
    return prompt

def get_answer_from_generated_text(text):
    parts = text.split('[INST]')
    question_answer = None
    for part in parts:
        if 'QUESTION :' in part:
            question_answer = part.strip()
            break

    if question_answer and '[/INST]' in question_answer:
        answer = question_answer.split('[/INST]')[1].strip()
        return "Answer: " + answer
    return "Couldn't fetch an answer, please try again."

def chatbot_function(message, state):
    if state is None:
        state = {"stage": "greeting"}

    if state["stage"] == "greeting":
        state["stage"] = "ask_for_article"
        return "Hello, welcome! Input an article to get started.", state

    if state["stage"] == "ask_for_article":
        state["article_text"] = message
        state["stage"] = "ask_question"
        chunks = chunk_text_by_sentences(message)
        summaries = [pipe(chunk)[0]['summary_text'] for chunk in chunks]
        state["summarized_article"] = " ".join(summaries)
        return "Article Summary : " + state["summarized_article"] + "\n\nYou can ask any questions based on the article.", state

    if state["stage"] == "ask_question":
        prompt = generate_prompt_for_qa(message, state["article_text"])
        input_ids = tokenizer_qa_bot.encode(prompt, return_tensors='pt')
        output = model_qa_bot.generate(input_ids, temperature=0.5)
        generated_text = tokenizer_qa_bot.decode(output[0])
        answer = get_answer_from_generated_text(generated_text)
        state["stage"] = "ask_to_continue"
        return answer + "\n\nDo you have any more questions? (Yes/No)", state

    if state["stage"] == "ask_to_continue":
        message = message.lower().strip()
        if message in ["yes", "y"]:
            state["stage"] = "ask_question"
            return "Great! Please enter your question.", state
        elif message in ["no", "n"]:
            return "Okay! It was nice talking to you!", state
        else:
            return "Sorry, I didn't catch that. Do you have any more questions? (Yes/No)", state


In [ ]:
import gradio as gr
css = """
    body { margin: 0; padding: 0; }
    .interface { width: 600px; margin: auto; }
    .input_textbox, .output_textbox { height: 400px; }
"""

iface = gr.Interface(
    fn=chatbot_function,
    inputs=[gr.Textbox(label="Your Message"), gr.State()],
    outputs=[gr.Textbox(label="Bot Response"), gr.State()],
    css=css
)

iface.launch(debug=True)